<a href="https://colab.research.google.com/github/charlyacha/labo1-colabs/blob/main/10_Ajuste_no_lineal_y_por_que_R2_miente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 10 — Ajuste no lineal: oscilador amortiguado, y por qué R² miente**Laboratorio 1 · Clase 10****Objetivos.**1. Ajustar un modelo **genuinamente no lineal** eligiendo las semillas a partir de la física.2. Diagnosticar el ajuste con el trío correcto: **$\\chi^2_\\nu$ + residuos + correlación de   parámetros**.3. Ver, con datos, **por qué $R^2$ no sirve para evaluar un modelo no lineal**.**Requisitos previos:** Colabs 06 y 09.Éste es el punto de llegada de toda la progresión estadística del curso.> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.optimize import curve_fitfrom scipy.signal import find_peaksfrom scipy import statsnp.random.seed(20261014)

---## 1. El modeloPara un oscilador subamortiguado,$$ x(t) = A\\, e^{-t/\\tau} \\cos(\\omega t + \\varphi) + x_0 $$Cinco parámetros, y —a diferencia de todo lo anterior— **el modelo no es lineal en ellos**. Esocambia tres cosas:1. No hay solución cerrada: `curve_fit` itera desde una semilla inicial `p0`.2. **Puede no converger**, o converger a un mínimo local sin avisar de manera evidente.3. Las incertezas de los parámetros salen de una aproximación lineal alrededor del mínimo, así que   son confiables solo si el mínimo es "bien portado".Por eso la semilla no se elige al azar: **se estima de la física del problema**.

In [ ]:
# --- DATOS DE EJEMPLO (reemplazar por los propios) ---A_r, tau_r, T_r, fase_r, off_r = 0.045, 3.2, 0.784, 0.35, 0.001fs = 200.0t = np.arange(0, 12, 1/fs)sigma_x = 0.0006x = (A_r*np.exp(-t/tau_r)*np.cos(2*np.pi*t/T_r + fase_r) + off_r     + np.random.normal(0, sigma_x, len(t)))err = np.full_like(t, sigma_x)# ------------------------------------------------------fig, ax = plt.subplots(figsize=(9, 3.4))ax.plot(t, x*1000, lw=0.7)ax.set_xlabel('$t$ [s]'); ax.set_ylabel('$x$ [mm]')ax.set_title('Oscilador amortiguado'); ax.grid(alpha=0.3)fig.tight_layout(); plt.show()

---## 2. Semillas a partir de la física- $x_0$: el promedio de la señal en la cola, cuando ya se amortiguó.- $A$: la amplitud del primer máximo, menos $x_0$.- $\\omega$: de los picos, con el método del Colab 09.- $\\tau$: del decaimiento de la envolvente — ajuste lineal de $\\ln(\\text{amplitud de los picos})$  contra el tiempo.- $\\varphi$: $0$ es casi siempre suficiente si el ajuste tiene los otros cuatro bien.

In [ ]:
# offsetx0_0 = x[int(0.9*len(x)):].mean()# picos -> período y envolventeidx, _ = find_peaks(x - x0_0, prominence=0.2*(x.max()-x0_0), distance=int(0.6*T_r*fs))tp, xp = t[idx], x[idx] - x0_0pT = np.polyfit(np.arange(len(tp)), tp, 1)T_0 = pT[0]w_0 = 2*np.pi/T_0A_0 = xp[0]ptau = np.polyfit(tp, np.log(xp), 1)      # ln A(t) = ln A0 - t/tautau_0 = -1/ptau[0]p0 = [A_0, tau_0, w_0, 0.0, x0_0]print(f"semillas:  A0 = {A_0:.4f} m   τ0 = {tau_0:.2f} s   "      f"ω0 = {w_0:.3f} rad/s   φ0 = 0   x0 = {x0_0:.5f} m")

In [ ]:
def amortiguado(t, A, tau, w, fase, x0):    return A*np.exp(-t/tau)*np.cos(w*t + fase) + x0popt, pcov = curve_fit(amortiguado, t, x, p0=p0, sigma=err,                       absolute_sigma=True, maxfev=20000)perr = np.sqrt(np.diag(pcov))nombres = ['A [m]', 'τ [s]', 'ω [rad/s]', 'φ [rad]', 'x₀ [m]']for n, v, e in zip(nombres, popt, perr):    print(f"{n:<12} {v:>10.5f} ± {e:<10.5f}  ({100*abs(e/v):.2f} %)")

> **Si no converge.** Los síntomas son: `RuntimeError: Optimal parameters not found`, una covarianza> llena de `inf`, o parámetros absurdos. En ese orden: (1) revisá las semillas —el 90 % de los casos> son ésas—; (2) subí `maxfev`; (3) usá `bounds=` para acotar los parámetros a valores físicamente> posibles (por ejemplo, $\\tau > 0$); (4) recortá el tramo de datos donde la señal ya se perdió en> el ruido, que no aporta información y sí molesta.

---## 3. El diagnóstico correcto

In [ ]:
def chi2_reducido(y, y_mod, sigma, p, verbose=True):    chi2 = np.sum(((y - y_mod)/sigma)**2)    nu = len(y) - p    if verbose:        print(f"χ² = {chi2:.1f}   ν = {nu}   χ²_ν = {chi2/nu:.3f}   "              f"p = {stats.chi2.sf(chi2, nu):.3f}")    return chi2/nudef matriz_correlacion(pcov):    d = np.sqrt(np.diag(pcov))    return pcov/np.outer(d, d)c2r = chi2_reducido(x, amortiguado(t, *popt), err, len(popt))print("\nmatriz de correlación de los parámetros:")M = matriz_correlacion(pcov)print("           " + "".join(f"{n.split()[0]:>10}" for n in nombres))for i, n in enumerate(nombres):    print(f"{n.split()[0]:>10} " + "".join(f"{M[i,j]:>10.2f}" for j in range(len(nombres))))

Mirá los elementos fuera de la diagonal. $A$ y $\\tau$ suelen estar fuertemente correlacionados: unaamplitud inicial mayor con un decaimiento más rápido describe casi los mismos datos. Eso tiene dosconsecuencias prácticas:- Reportar $A$ y $\\tau$ por separado con sus errores **pierde información**: sus incertezas no son  independientes.- Si vas a combinarlos en una cuenta posterior, la propagación del Colab 04 (que supone  independencia) **no aplica**; hay que incluir el término cruzado  $2\\,\\frac{\\partial f}{\\partial A}\\frac{\\partial f}{\\partial \\tau}\\,\\mathrm{cov}(A,\\tau)$.Éste es exactamente el problema que aparece al ajustar espectros de impedancia o curvas I–V convarios parámetros: la covarianza no es un tecnicismo, es la diferencia entre una barra de errorhonesta y una inventada.

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True,                             gridspec_kw={'height_ratios': [2, 1]})a1.plot(t, x*1000, '.', ms=1.8, alpha=0.5, label='datos')a1.plot(t, amortiguado(t, *popt)*1000, 'crimson', lw=1.2, label='ajuste')env = popt[0]*np.exp(-t/popt[1])*1000a1.plot(t, popt[4]*1000 + env, 'k--', lw=0.8, alpha=0.6, label='envolvente')a1.plot(t, popt[4]*1000 - env, 'k--', lw=0.8, alpha=0.6)a1.set_ylabel('$x$ [mm]'); a1.grid(alpha=0.3); a1.legend(ncol=3, fontsize=9)a1.set_title(f'Ajuste no lineal — $\\chi^2_\\nu$ = {c2r:.2f}')a2.plot(t, (x - amortiguado(t, *popt))/err, '.', ms=1.8, alpha=0.5)a2.axhline(0, color='crimson', lw=1)for s in (-2, 2): a2.axhline(s, color='gray', lw=0.7, ls=':')a2.set_xlabel('$t$ [s]'); a2.set_ylabel('residuo / σ'); a2.grid(alpha=0.3)fig.subplots_adjust(hspace=0.08); plt.show()

Graficar los residuos **normalizados por la incerteza** (`residuo/σ`) es un buen hábito: la escalapasa a ser universal y esperás que el 95 % de los puntos caiga dentro de $\\pm 2$. Si ves mucho másafuera, subestimaste $\\sigma$; si ves casi todos adentro de $\\pm 1$, la sobreestimaste.

---## 4. Por qué R² miente$R^2$ se calcula convencionalmente como$$ R^2 = 1 - \\frac{\\sum (y_i - f_i)^2}{\\sum (y_i - \\bar{y})^2} $$es decir, comparando el modelo contra "usar el promedio de $y$ como predicción". Esa comparacióntiene sentido en una regresión lineal ordinaria. **Fuera de ese contexto, no la tiene**, y por dosmotivos distintos que conviene separar.

In [ ]:
def R2(y, y_mod):    y = np.asarray(y, float)    return 1 - np.sum((y - y_mod)**2)/np.sum((y - y.mean())**2)

### Problema 1 — R² no distingue un modelo correcto de uno incorrectoGeneramos datos de un decaimiento exponencial y los ajustamos con dos modelos: el correcto y unofrancamente equivocado (una hipérbola).

In [ ]:
tt = np.linspace(0.2, 6, 80)sg = 0.006yy = 1.0*np.exp(-tt/1.8) + 0.05 + np.random.normal(0, sg, len(tt))ee = np.full_like(tt, sg)def modelo_correcto(t, A, tau, c):  return A*np.exp(-t/tau) + cdef modelo_malo(t, A, b, c):        return A/(1 + t/b) + cpc, _ = curve_fit(modelo_correcto, tt, yy, p0=[1, 2, 0], sigma=ee, absolute_sigma=True)pm, _ = curve_fit(modelo_malo,     tt, yy, p0=[1, 2, 0], sigma=ee, absolute_sigma=True)print(f"{'modelo':<24}{'R²':>10}{'χ²_ν':>10}")print("-"*44)for nom, f, p in [('exponencial (correcto)', modelo_correcto, pc),                  ('hipérbola (incorrecto)', modelo_malo, pm)]:    r2 = R2(yy, f(tt, *p))    c2 = np.sum(((yy - f(tt, *p))/ee)**2)/(len(tt)-3)    print(f"{nom:<24}{r2:>10.4f}{c2:>10.2f}")

Los dos $R^2$ son altísimos y **prácticamente indistinguibles**: si informaras solo ese número,nadie podría decir cuál modelo es el correcto. El $\\chi^2_\\nu$ los separa sin ambigüedad, y losresiduos lo muestran a simple vista:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 5.6), sharex=True,                         gridspec_kw={'height_ratios': [2, 1]})for col, (nom, f, p) in enumerate([('exponencial (correcto)', modelo_correcto, pc),                                   ('hipérbola (incorrecto)', modelo_malo, pm)]):    a1, a2 = axes[0, col], axes[1, col]    a1.errorbar(tt, yy, yerr=ee, fmt='o', ms=4, capsize=2, alpha=0.7)    a1.plot(tt, f(tt, *p), 'crimson', lw=1.6)    a1.set_title(f'{nom}\n$R^2$ = {R2(yy, f(tt,*p)):.4f}', fontsize=10)    a1.grid(alpha=0.3)    a2.plot(tt, (yy - f(tt, *p))/ee, 'o', ms=4)    a2.axhline(0, color='crimson', lw=1)    a2.set_xlabel('t'); a2.set_ylabel('residuo / σ'); a2.grid(alpha=0.3)axes[0,0].set_ylabel('y')fig.subplots_adjust(hspace=0.1); plt.show()

Los residuos del modelo malo dibujan una curva sistemática que sube, baja y vuelve a subir. Los delmodelo correcto son ruido. **Un solo gráfico resuelve lo que $R^2$ no podía.**

### Problema 2 — R² depende del rango muestreado, aunque el modelo sea correctoÉste es el más traicionero. Tomamos el **mismo modelo correcto**, el **mismo ruido**, y solocambiamos el rango de $t$ que medimos.

In [ ]:
sg2 = 0.02      # ruido algo mayor, para que el efecto se vea en pocas cifrasprint(f"{'rango de t':<16}{'R²':>10}{'χ²_ν':>10}   (mismo modelo correcto en los tres casos)")print("-"*56)for tmax, etiqueta in [(6.0, '0,2 a 6,0'), (2.0, '0,2 a 2,0'),                       (0.8, '0,2 a 0,8'), (0.5, '0,2 a 0,5')]:    ts = np.linspace(0.2, tmax, 40)    ys = 1.0*np.exp(-ts/1.8) + 0.05 + np.random.normal(0, sg2, len(ts))    es = np.full_like(ts, sg2)    ps, _ = curve_fit(modelo_correcto, ts, ys, p0=[1, 2, 0], sigma=es, absolute_sigma=True)    r2 = R2(ys, modelo_correcto(ts, *ps))    c2 = np.sum(((ys - modelo_correcto(ts, *ps))/es)**2)/(len(ts)-3)    print(f"{etiqueta:<16}{r2:>10.4f}{c2:>10.2f}")

El $R^2$ cae de 0,99 a 0,9 y pico al achicar el rango, aunque el modelo sea **exactamente elcorrecto** y el ruido sea **exactamente el mismo**. La razón es estructural: $R^2$ compara contra la varianza totalde los datos, y si medís un tramo chico donde $y$ apenas varía, esa varianza total es pequeña ycualquier modelo "explica poco". El $\\chi^2_\\nu$, en cambio, se mantiene alrededor de 1 en los trescasos, porque compara contra la incerteza experimental, que es lo que corresponde.Esto está documentado formalmente en:> A.-N. Spiess & N. Neumeyer, *"An evaluation of R² as an inadequate measure for nonlinear models in> pharmacological and biochemical research: a Monte Carlo approach"*, **BMC Pharmacology** 10:6> (2010).Los autores muestran con datos simulados que $R^2$ puede ser alto para un modelo francamenteincorrecto y bajo para el correcto, según el ruido y el rango muestreado. Es un problema prácticorecurrente en cualquier disciplina que ajuste modelos no lineales.

### Resumen operativo| Herramienta | ¿Sirve para bondad de ajuste no lineal? ||---|---|| $R$, $R^2$ | **No.** Solo describe alineación lineal; depende del rango || $\\chi^2_\\nu$ con $\\sigma_i$ reales | **Sí**, es el indicador principal || Gráfico de residuos | **Sí**, y detecta lo que ningún número resume || Errores de parámetros (`pcov`) | **Sí**, un parámetro sin incerteza no es un resultado || Matriz de correlación | Necesaria si vas a combinar parámetros || AIC / BIC | Solo para comparar modelos no anidados. Optativo, avanzado |

---## 5. De los parámetros a la físicaDos cuentas, para ver cuándo la covarianza importa y cuándo no.- $\omega_0 = \sqrt{\omega^2 + 1/\tau^2}$ combina $\omega$ y $\tau$, que están **poco**  correlacionados: propagar como independientes es aceptable.- La amplitud de la envolvente a un tiempo dado, $A\,e^{-t_1/\tau}$, combina $A$ y $\tau$, que están  **fuertemente anticorrelados**: ignorar la covarianza infla la barra de error de manera apreciable.La moraleja no es "siempre hay que usar la matriz completa", sino **mirar $\rho$ antes de decidir**.

In [ ]:
A_f, tau_f, w_f, fase_f, x0_f = poptdA, dtau, dw, dfase, dx0 = perr# --- frecuencia natural:  ω0² = ω² + 1/τ²  ---w0 = np.sqrt(w_f**2 + 1/tau_f**2)dw0 = np.sqrt((w_f/w0*dw)**2 + (dtau/(w0*tau_f**3))**2)Q = w0*tau_f/2dQ = Q*np.sqrt((dw0/w0)**2 + (dtau/tau_f)**2)print(f"ω  = {w_f:.4f} ± {dw:.4f} rad/s")print(f"τ  = {tau_f:.4f} ± {dtau:.4f} s")print(f"ω₀ = {w0:.4f} ± {dw0:.4f} rad/s")print(f"Q  = {Q:.2f} ± {dQ:.2f}")# --- amplitud de la envolvente a t = 5 s:  A e^(-t/τ)  ---# A y τ están fuertemente anticorrelados, así que ACÁ la covarianza sí importat1 = 5.0env = A_f*np.exp(-t1/tau_f)dEdA   = np.exp(-t1/tau_f)dEdtau = A_f*np.exp(-t1/tau_f)*t1/tau_f**2cov_At = pcov[0, 1]sin_cov = np.sqrt((dEdA*dA)**2 + (dEdtau*dtau)**2)con_cov = np.sqrt((dEdA*dA)**2 + (dEdtau*dtau)**2 + 2*dEdA*dEdtau*cov_At)print(f"\nAmplitud de la envolvente a t = {t1:.0f} s:")print(f"  ρ(A, τ) = {cov_At/(dA*dtau):+.2f}")print(f"  σ ignorando la covarianza : {sin_cov*1e6:.2f} µm")print(f"  σ incluyéndola            : {con_cov*1e6:.2f} µm")print(f"  sobrestimación si se ignora: {100*(sin_cov/con_cov - 1):.0f} %")

---## 6. Ejercicios**10.1.** Ajustá tus propios datos del oscilador amortiguado. Reportá $\\tau$ y $\\omega$ con susincertezas, el $\\chi^2_\\nu$, el gráfico con residuos normalizados y la matriz de correlación.Esto es exactamente lo que se evalúa en el Informe 4.**10.2.** Corré el ajuste con semillas deliberadamente malas (`p0=[1, 1, 1, 0, 0]`). ¿Converge? ¿Aqué? Es importante que veas al menos una vez cómo se ve un ajuste que falló pero devuelve números.**10.3.** Ajustá tus datos con un modelo **sin** amortiguamiento ($A\\cos(\\omega t+\\varphi)+x_0$).Calculá $R^2$ y $\\chi^2_\\nu$ de ambos ajustes. ¿Cuál de los dos indicadores te habría alertado?**10.4.** Recortá tus datos a los primeros dos períodos y volvé a ajustar. ¿Qué le pasa a$\\sigma_\\tau$? ¿Y a $R^2$? Relacionalo con el Problema 2 de la sección 4.**10.5.** *(avanzado, optativo)* Investigá qué son el AIC y el BIC y en qué se diferencian del testde modelos anidados del Colab 08. ¿Cuál usarías para decidir entre un oscilador subamortiguado y unosobreamortiguado?